# 🇫🇷 Exploration des données PPA France

Ce notebook explore et visualise toutes les données générées par `download_france_data.py`.

**Prérequis :** Avoir exécuté `download_france_data.py` au préalable.

```bash
pip install plotly nbformat openpyxl
```

In [16]:
import pandas as pd
import numpy as np
import os
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Chemin vers votre dossier database (à adapter si nécessaire)
DB_PATH = "../French_PPA/database"  # relatif depuis votre notebook
# Ou chemin absolu :
# DB_PATH = r"C:/Users/cypri/OneDrive/Documents/1. Pro/Formations/Jupyter Notebooks/pyPPA/French_PPA/database"

---
## 1. 📊 Grid France — CO2, Mix ENR, CAPEX
**Fichier :** `grid_france.csv`  
**Statut :** 🟡 Synthétique (projections ADEME/RTE)

In [17]:
print(os.getcwd())

c:\Users\cypri\OneDrive\Documents\1. Pro\Formations\Jupyter Notebooks\pyPPA\French_PPA


In [18]:
grid_df = pd.read_csv(f"{DB_PATH}/grid_france.csv", index_col=0)
print("Aperçu grid_france.csv :")
print(f"   Années : {grid_df.index.min()} → {grid_df.index.max()}")
display(grid_df.head(10))

Aperçu grid_france.csv :
   Années : 2023 → 2050


,solar_capex,wind_capex,co2,ren_share
year,,,,
2023,700000.0,3000000.0,51.50,0.2650
2024,682857.0,2928571.0,47.20,0.2900
2025,665714.0,2857143.0,45.77,0.3012
2026,648571.0,2785714.0,44.34,0.3123
2027,631429.0,2714286.0,42.91,0.3235
2028,614286.0,2642857.0,41.48,0.3346
2029,597143.0,2571429.0,40.05,0.3458
2030,580000.0,2500000.0,38.62,0.3569
2031,567000.0,2450000.0,37.18,0.3681


In [4]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Intensité CO2 du réseau (gCO2/kWh)",
        "Part ENR dans le mix (%)",
        "CAPEX Solaire (k€/MW)",
        "CAPEX Éolien Offshore (k€/MW)"
    ]
)

# CO2
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['co2'],
    name='CO2 (gCO2/kWh)', line=dict(color='#e74c3c', width=2),
    fill='tozeroy', fillcolor='rgba(231,76,60,0.1)'
), row=1, col=1)

# ENR share
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['ren_share'] * 100,
    name='ENR (%)', line=dict(color='#27ae60', width=2),
    fill='tozeroy', fillcolor='rgba(39,174,96,0.1)'
), row=1, col=2)

# CAPEX Solaire
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['solar_capex'] / 1000,
    name='CAPEX PV (k€/MW)', line=dict(color='#f39c12', width=2)
), row=2, col=1)

# CAPEX Éolien
fig.add_trace(go.Scatter(
    x=grid_df.index, y=grid_df['wind_capex'] / 1000,
    name='CAPEX Éolien (k€/MW)', line=dict(color='#2980b9', width=2)
), row=2, col=2)

fig.update_layout(
    title="📈 Projections Grid France 2023–2050",
    height=600, showlegend=False,
    template='plotly_white'
)
fig.show()

print(f"\n💡 Note : CO2 France (55 gCO2/kWh) vs Corée (~450 gCO2/kWh)")
print(f"   La France est déjà très décarbonée grâce au nucléaire.")


💡 Note : CO2 France (55 gCO2/kWh) vs Corée (~450 gCO2/kWh)
   La France est déjà très décarbonée grâce au nucléaire.


---
## 2. ☀️ Profils Solaires PVGIS
**Fichier :** `solar_patterns.db`  
**Statut :** 🟢 Données RÉELLES (PVGIS JRC, Dunkerque 2020)

In [5]:
conn = sqlite3.connect(f"{DB_PATH}/solar_patterns.db")

# Lister les tables disponibles
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"📋 Tables dans solar_patterns.db : {tables['name'].tolist()}")

solar_df = pd.read_sql("SELECT * FROM solar_patterns", conn)
conn.close()

solar_df['datetime'] = pd.to_datetime(solar_df['datetime'])
solar_df = solar_df.set_index('datetime')

print(f"\n📊 Statistiques du profil solaire (Dunkerque 2020) :")
print(f"   Nombre d'heures : {len(solar_df):,}")
print(f"   CF moyen annuel : {solar_df['q99'].mean():.3f} ({solar_df['q99'].mean()*100:.1f}%)")
print(f"   CF max (heure de pointe) : {solar_df['q99'].max():.3f}")
print(f"   Heures de production > 10% : {(solar_df['q99'] > 0.1).sum():,}")
display(solar_df.describe())

📋 Tables dans solar_patterns.db : ['solar_patterns']

📊 Statistiques du profil solaire (Dunkerque 2020) :
   Nombre d'heures : 8,784
   CF moyen annuel : 0.130 (13.0%)
   CF max (heure de pointe) : 0.889
   Heures de production > 10% : 2,693


,q99
count,8784.000000
mean,0.130387
std,0.218699
min,0.000000
25%,0.000000
50%,0.000000
75%,0.172387
max,0.888530


In [6]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Profil horaire — Semaine type été (Juin)",
        "Profil horaire — Semaine type hiver (Décembre)",
        "Distribution mensuelle du CF solaire",
        "Heatmap solaire (heure × mois)"
    ]
)

# Semaine type été
ete = solar_df[solar_df.index.month == 6].iloc[:168]  # 1 semaine
fig.add_trace(go.Scatter(
    x=list(range(len(ete))), y=ete['q99'],
    name='Été (Juin)', line=dict(color='#f39c12'), fill='tozeroy'
), row=1, col=1)

# Semaine type hiver
hiver = solar_df[solar_df.index.month == 12].iloc[:168]
fig.add_trace(go.Scatter(
    x=list(range(len(hiver))), y=hiver['q99'],
    name='Hiver (Décembre)', line=dict(color='#2980b9'), fill='tozeroy'
), row=1, col=2)

# Distribution mensuelle
monthly_cf = solar_df.groupby(solar_df.index.month)['q99'].mean() * 100
month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
               'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
fig.add_trace(go.Bar(
    x=month_names, y=monthly_cf.values,
    name='CF mensuel (%)',
    marker_color=['#2980b9']*3 + ['#f39c12']*6 + ['#2980b9']*3
), row=2, col=1)

# Heatmap heure × mois
pivot = solar_df.copy()
pivot['hour'] = pivot.index.hour
pivot['month'] = pivot.index.month
heatmap_data = pivot.groupby(['hour', 'month'])['q99'].mean().unstack()

fig.add_trace(go.Heatmap(
    z=heatmap_data.values,
    x=month_names,
    y=[f"{h}h" for h in range(24)],
    colorscale='YlOrRd',
    showscale=True,
    name='Heatmap CF'
), row=2, col=2)

fig.update_layout(
    title="☀️ Profil Solaire PVGIS — Dunkerque 2020 (Données RÉELLES)",
    height=700, showlegend=False,
    template='plotly_white'
)
fig.show()

---
## 3. 💨 Profils Éoliens Open-Meteo/ERA5
**Fichier :** `wind_patterns.db`  
**Statut :** 🟢 Données RÉELLES (ERA5 réanalyse ECMWF)

In [7]:
conn = sqlite3.connect(f"{DB_PATH}/wind_patterns.db")
tables_wind = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"📋 Tables dans wind_patterns.db : {tables_wind['name'].tolist()}")

wind_df = pd.read_sql("SELECT * FROM wind_patterns", conn)
conn.close()

wind_df['index'] = pd.to_datetime(wind_df['index'])
wind_df = wind_df.set_index('index')

print(f"\n📊 Statistiques par région :")
print(wind_df.describe().round(3))

📋 Tables dans wind_patterns.db : ['wind_patterns']

📊 Statistiques par région :
       Hauts-de-France  Normandie  Bretagne  Pays de la Loire  Occitanie
count         8784.000   8784.000  8784.000          8784.000   8784.000
mean             0.252      0.316     0.227             0.392      0.407
std              0.332      0.366     0.313             0.397      0.421
min              0.000      0.000     0.000             0.000      0.000
25%              0.008      0.012     0.008             0.027      0.008
50%              0.088      0.136     0.076             0.222      0.210
75%              0.365      0.563     0.299             0.831      1.000
max              1.000      1.000     1.000             1.000      1.000


In [8]:
regions = [c for c in wind_df.columns]
colors = ['#2980b9', '#27ae60', '#e74c3c', '#f39c12', '#9b59b6']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "CF moyen mensuel par région",
        "Distribution des CF (boîte à moustaches)",
        "Profil éolien — Semaine type (Janvier)",
        "Courbe de durée éolienne annuelle"
    ]
)

# CF mensuel par région
for i, (region, color) in enumerate(zip(regions, colors)):
    monthly = wind_df[region].groupby(wind_df.index.month).mean() * 100
    fig.add_trace(go.Scatter(
        x=month_names, y=monthly.values,
        name=region, line=dict(color=color, width=2),
        mode='lines+markers'
    ), row=1, col=1)

# Boxplot distribution
for i, (region, color) in enumerate(zip(regions, colors)):
    fig.add_trace(go.Box(
        y=wind_df[region] * 100,
        name=region,
        marker_color=color,
        showlegend=False
    ), row=1, col=2)

# Semaine type janvier
janvier = wind_df[wind_df.index.month == 1].iloc[:168]
for i, (region, color) in enumerate(zip(regions, colors)):
    fig.add_trace(go.Scatter(
        x=list(range(len(janvier))), y=janvier[region],
        name=region, line=dict(color=color),
        showlegend=False
    ), row=2, col=1)

# Courbe de durée
for i, (region, color) in enumerate(zip(regions, colors)):
    sorted_cf = np.sort(wind_df[region].values)[::-1] * 100
    hours = np.arange(1, len(sorted_cf) + 1)
    fig.add_trace(go.Scatter(
        x=hours, y=sorted_cf,
        name=region, line=dict(color=color),
        showlegend=False
    ), row=2, col=2)

fig.update_layout(
    title="💨 Profils Éoliens Offshore — France 2020 (Données RÉELLES ERA5)",
    height=700, template='plotly_white',
    legend=dict(orientation='h', y=-0.15)
)
fig.update_yaxes(title_text="CF (%)", row=1, col=1)
fig.update_yaxes(title_text="CF (%)", row=1, col=2)
fig.update_xaxes(title_text="Heures (rang)", row=2, col=2)
fig.show()

# Tableau récapitulatif
print("\n📊 Capacité Factor annuel moyen par région :")
summary = pd.DataFrame({
    'CF moyen (%)': (wind_df.mean() * 100).round(1),
    'CF max (%)': (wind_df.max() * 100).round(1),
    'Heures > 50%': (wind_df > 0.5).sum(),
    'Heures > 80%': (wind_df > 0.8).sum(),
})
display(summary)


📊 Capacité Factor annuel moyen par région :


,CF moyen (%),CF max (%),Heures > 50%,Heures > 80%
Hauts-de-France,25.2,100.0,1698,1154
Normandie,31.6,100.0,2394,1592
Bretagne,22.7,100.0,1512,942
Pays de la Loire,39.2,100.0,3080,2270
Occitanie,40.7,100.0,3307,2639


---
## 4. 🏭 Profil de Charge Industriel
**Fichier :** `load_patterns.db`  
**Statut :** 🟡 Synthétique (profil semi-continu générique)

In [9]:
conn = sqlite3.connect(f"{DB_PATH}/load_patterns.db")
load_df = pd.read_sql("SELECT * FROM load_patterns", conn)
conn.close()

load_df['datetime'] = pd.to_datetime(load_df['datetime'])
load_df = load_df.set_index('datetime')

print(f"📊 Profil de charge industriel :")
print(f"   CF moyen : {load_df['value'].mean():.3f} ({load_df['value'].mean()*100:.1f}%)")
print(f"   CF min : {load_df['value'].min():.3f}")
print(f"   CF max : {load_df['value'].max():.3f}")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Profil type 2 semaines (Janv-Fév)", "Distribution des charges"]
)

# 2 semaines
two_weeks = load_df.iloc[:336]
fig.add_trace(go.Scatter(
    x=two_weeks.index, y=two_weeks['value'] * 100,
    fill='tozeroy', line=dict(color='#8e44ad'),
    name='Charge (%)'
), row=1, col=1)

# Histogramme
fig.add_trace(go.Histogram(
    x=load_df['value'] * 100,
    nbinsx=30, marker_color='#8e44ad',
    name='Distribution'
), row=1, col=2)

fig.update_layout(
    title="🏭 Profil de Charge Industriel (Synthétique — À remplacer par données client)",
    height=400, template='plotly_white', showlegend=False
)
fig.show()

print("\n⚠️  Ce profil est SYNTHÉTIQUE.")
print("   Demandez à votre client sa courbe de charge annuelle réelle")
print("   (disponible via son gestionnaire énergie ou son contrat ENEDIS).")

📊 Profil de charge industriel :
   CF moyen : 0.895 (89.5%)
   CF min : 0.637
   CF max : 1.000



⚠️  Ce profil est SYNTHÉTIQUE.
   Demandez à votre client sa courbe de charge annuelle réelle
   (disponible via son gestionnaire énergie ou son contrat ENEDIS).


---
## 5. ⚡ Tarifs Réseau TURPE (KEPCO_france.xlsx)
**Fichier :** `KEPCO_france.xlsx`  
**Statut :** 🟡 Synthétique (valeurs CRE approximées)

In [10]:
# Lire toutes les sheets
xl = pd.ExcelFile(f"{DB_PATH}/TURPE_france.xlsx")
print(f"📋 Sheets disponibles : {xl.sheet_names}")

for sheet in xl.sheet_names:
    df = pd.read_excel(f"{DB_PATH}/TURPE_france.xlsx", sheet_name=sheet)
    print(f"\n--- Sheet '{sheet}' ---")
    # display(df)

📋 Sheets disponibles : ['README', 'timezone', 'season', 'contract', 'HTB3', 'HTB2', 'HTB1']

--- Sheet 'README' ---

--- Sheet 'timezone' ---

--- Sheet 'season' ---

--- Sheet 'contract' ---

--- Sheet 'HTB3' ---

--- Sheet 'HTB2' ---

--- Sheet 'HTB1' ---


In [11]:
# Reconstituer le profil tarifaire horaire TURPE pour 2030
import sys
sys.path.insert(0, '..')  # Ajouter le dossier parent si FranceGridUtils est là

# Construction manuelle du profil pour visualisation
year = 2030
date_range = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31 23:00", freq="h")

months = date_range.month
hours = date_range.hour
weekdays = date_range.weekday

is_winter = months.isin([11, 12, 1, 2, 3])
is_daytime = (hours >= 6) & (hours < 22)
is_weekday = weekdays < 5

# TURPE énergie
turpe_energy = np.zeros(len(date_range))
turpe_energy[is_winter & is_daytime & is_weekday] = 5.2   # HPH
turpe_energy[is_winter & ~(is_daytime & is_weekday)] = 1.8  # HCH
turpe_energy[~is_winter & is_daytime & is_weekday] = 1.0  # HPE
turpe_energy[~is_winter & ~(is_daytime & is_weekday)] = 0.5  # HCE

# EPEX synthétique (80 €/MWh moyen)
base = 80.0
seasonal = np.where(months.isin([12,1,2]), 1.30, np.where(months.isin([6,7,8]), 0.90, 1.0))
np.random.seed(42)
epex = base * seasonal * np.random.lognormal(0, 0.1, len(date_range))

total_rate = epex + turpe_energy + 0.5 + 0.3  # + TICFE + CTA

rate_df = pd.DataFrame({
    'datetime': date_range,
    'EPEX Spot': epex,
    'TURPE Énergie': turpe_energy,
    'Taxes (TICFE+CTA)': 0.8,
    'Total (€/MWh)': total_rate
}).set_index('datetime')

# Subplot specs : (2,2) doit être 'domain' pour accueillir un Pie
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Profil tarifaire — Semaine type hiver",
        "Profil tarifaire — Semaine type été",
        "Distribution mensuelle du tarif total",
        "Composition du tarif par composante"
    ],
    specs=[
        [{"type": "xy"},     {"type": "xy"}],
        [{"type": "xy"},     {"type": "domain"}]   # domain requis pour go.Pie
    ]
)

# Hiver
hiver_rate = rate_df[rate_df.index.month == 1].iloc[:168]
fig.add_trace(go.Scatter(x=list(range(168)), y=hiver_rate['Total (€/MWh)'],
    name='Tarif total', line=dict(color='#e74c3c')), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(168)), y=hiver_rate['EPEX Spot'],
    name='EPEX Spot', line=dict(color='#e74c3c', dash='dot')), row=1, col=1)

# Été
ete_rate = rate_df[rate_df.index.month == 7].iloc[:168]
fig.add_trace(go.Scatter(x=list(range(168)), y=ete_rate['Total (€/MWh)'],
    name='Tarif total été', line=dict(color='#f39c12'), showlegend=False), row=1, col=2)

# Monthly bar
monthly_rate = rate_df.groupby(rate_df.index.month)['Total (€/MWh)'].mean()
fig.add_trace(go.Bar(
    x=month_names, y=monthly_rate.values,
    marker_color=['#2980b9']*3 + ['#27ae60']*6 + ['#2980b9']*3,
    showlegend=False
), row=2, col=1)

# Composition — go.Pie dans subplot type='domain'
components = ['EPEX Spot', 'TURPE Énergie', 'Taxes (TICFE+CTA)']
values = [rate_df['EPEX Spot'].mean(), rate_df['TURPE Énergie'].mean(), 0.8]
fig.add_trace(go.Pie(
    labels=components, values=values,
    hole=0.4,
    marker_colors=['#e74c3c', '#2980b9', '#27ae60']
), row=2, col=2)

fig.update_layout(
    title="⚡ Structure du Tarif Électricité HTB2 — France 2030",
    height=650, template='plotly_white'
)
fig.show()

print(f"\n📊 Résumé tarifaire 2030 (€/MWh) :")
print(f"   Tarif total moyen : {total_rate.mean():.1f} €/MWh")
print(f"   EPEX moyen        : {epex.mean():.1f} €/MWh")
print(f"   TURPE énergie moy : {turpe_energy.mean():.1f} €/MWh")
print(f"   Tarif min/max     : {total_rate.min():.1f} / {total_rate.max():.1f} €/MWh")


📊 Résumé tarifaire 2030 (€/MWh) :
   Tarif total moyen : 87.0 €/MWh
   EPEX moyen        : 84.4 €/MWh
   TURPE énergie moy : 1.8 €/MWh
   Tarif min/max     : 53.7 / 158.9 €/MWh


---
## 6. 🌬️ Parcs Éoliens Offshore (wind_grid_france.xlsx)
**Fichier :** `wind_grid_france.xlsx`  
**Statut :** 🟡 Synthétique (6 parcs encodés manuellement)

In [12]:
wind_grid = pd.read_excel(f"{DB_PATH}/wind_grid_france.xlsx")
print("📋 Parcs éoliens offshore français :")
display(wind_grid)

# Carte des parcs
fig = px.scatter_mapbox(
    wind_grid,
    lat='lat', lon='lon',
    size='capacity',
    color='LCOE',
    hover_name='nom_parc',
    hover_data={'capacity': True, 'LCOE': True, 'DT_m': True, 'admin_boundaries': True},
    color_continuous_scale='RdYlGn_r',
    size_max=30,
    zoom=5,
    center={'lat': 48.5, 'lon': -1.0},
    mapbox_style='open-street-map',
    title='🌬️ Parcs Éoliens Offshore France — LCOE (€/MWh) et Capacité'
)
fig.update_layout(height=500)
fig.show()

# Comparatif LCOE
fig2 = px.bar(
    wind_grid.sort_values('LCOE'),
    x='nom_parc', y='LCOE',
    color='admin_boundaries',
    text='capacity',
    title='Comparatif LCOE des parcs éoliens offshore (€/MWh)',
    labels={'nom_parc': 'Parc', 'LCOE': 'LCOE (€/MWh)', 'capacity': 'Capacité (MW)'}
)
fig2.update_traces(texttemplate='%{text} MW', textposition='outside')
fig2.update_layout(template='plotly_white', height=400)
fig2.show()

📋 Parcs éoliens offshore français :


,nom_parc,admin_boundaries,LCOE,rec_weight,capacity,DT_m,lat,lon
0,Saint-Nazaire,Pays de la Loire,65,1,480,12000,47.2,-2.5
1,Fécamp,Normandie,70,1,500,15000,49.8,0.5
2,Courseulles-sur-Mer,Normandie,72,1,450,10000,49.4,-0.5
3,Saint-Brieuc,Bretagne,75,1,496,16000,48.6,-2.8
4,Dunkerque,Hauts-de-France,60,1,600,10000,51.1,2.2
5,Golfe du Lion,Occitanie,80,1,250,20000,43.1,4.2


---
## 7. 🔄 Vue d'ensemble : Comparaison Solaire vs Éolien
Superposition des profils pour comprendre la complémentarité des sources

In [13]:
# Recharger les données
conn_s = sqlite3.connect(f"{DB_PATH}/solar_patterns.db")
solar = pd.read_sql("SELECT * FROM solar_patterns", conn_s)
solar['datetime'] = pd.to_datetime(solar['datetime'])
solar = solar.set_index('datetime')
conn_s.close()

conn_w = sqlite3.connect(f"{DB_PATH}/wind_patterns.db")
wind = pd.read_sql("SELECT * FROM wind_patterns", conn_w)
wind['index'] = pd.to_datetime(wind['index'])
wind = wind.set_index('index')
conn_w.close()

# ── Alignement : PVGIS retourne des timestamps à :10 (ex: 00:10, 01:10...)
#    Open-Meteo retourne des timestamps à :00. On aligne en arrondissant au bas de l'heure.
solar.index = solar.index.floor('h')
solar = solar[~solar.index.duplicated(keep='first')]   # sécurité si doublons après arrondi
wind  = wind[~wind.index.duplicated(keep='first')]

common_idx = solar.index.intersection(wind.index)
print(f"✅ Index commun : {len(common_idx)} heures")

solar_aligned = solar.loc[common_idx, 'q99']
best_wind = wind.loc[common_idx].mean(axis=1)  # moyenne toutes régions (plus stable que max)
# Exclure colonnes Occitanie si toutes à 0 (données manquantes)
valid_cols = [c for c in wind.columns if wind[c].mean() > 0.01]
best_wind = wind.loc[common_idx, valid_cols].mean(axis=1)

# Semaine type hiver / été
week_winter = common_idx[(common_idx.month == 1)][:168]
week_summer = common_idx[(common_idx.month == 7)][:168]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Complémentarité Solaire/Éolien — Hiver (1 semaine)",
        "Complémentarité Solaire/Éolien — Été (1 semaine)",
        "CF mensuel moyen — Comparaison sources",
        "Corrélation Solaire vs Éolien"
    ]
)

for week, r_idx, season in [(week_winter, 1, 'Hiver'), (week_summer, 1, 'Été')]:
    c_idx = 1 if season == 'Hiver' else 2
    show_leg = (season == 'Hiver')
    fig.add_trace(go.Scatter(
        x=list(range(len(week))), y=solar_aligned.loc[week].values * 100,
        name='Solaire', line=dict(color='#f39c12', width=2),
        fill='tozeroy', fillcolor='rgba(243,156,18,0.2)',
        showlegend=show_leg
    ), row=1, col=c_idx)
    fig.add_trace(go.Scatter(
        x=list(range(len(week))), y=best_wind.loc[week].values * 100,
        name='Éolien (moy. régions)', line=dict(color='#2980b9', width=2),
        fill='tozeroy', fillcolor='rgba(41,128,185,0.2)',
        showlegend=show_leg
    ), row=1, col=c_idx)

# CF mensuel moyen
monthly_solar = solar_aligned.groupby(solar_aligned.index.month).mean() * 100
monthly_wind  = best_wind.groupby(best_wind.index.month).mean() * 100

fig.add_trace(go.Scatter(
    x=month_names, y=monthly_solar.values,
    name='Solaire', line=dict(color='#f39c12', width=3),
    mode='lines+markers', showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=month_names, y=monthly_wind.values,
    name='Éolien', line=dict(color='#2980b9', width=3),
    mode='lines+markers', showlegend=False
), row=2, col=1)

# Scatter corrélation — sécurité si données insuffisantes
n_sample = min(2000, len(solar_aligned))
if n_sample > 10:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(solar_aligned), n_sample, replace=False)
    fig.add_trace(go.Scatter(
        x=solar_aligned.iloc[sample_idx].values * 100,
        y=best_wind.iloc[sample_idx].values * 100,
        mode='markers',
        marker=dict(size=3, opacity=0.3, color='#27ae60'),
        name='Corrélation',
        showlegend=False
    ), row=2, col=2)
    corr = np.corrcoef(solar_aligned.values, best_wind.values)[0, 1]
else:
    corr = float('nan')
    print("⚠️ Pas assez de données pour calculer la corrélation")

fig.update_layout(
    title=f"🔄 Complémentarité Solaire/Éolien (corrélation = {corr:.3f})",
    height=700, template='plotly_white'
)
fig.update_xaxes(title_text="Solaire CF (%)", row=2, col=2)
fig.update_yaxes(title_text="Éolien CF (%)", row=2, col=2)
fig.show()

print(f"\n💡 Corrélation solaire/éolien : {corr:.3f}")
print(f"   Valeur proche de 0 = pas de lien fort (normal solaire vs éolien)")
print(f"   Hiver = éolien fort + solaire faible → complémentaires ✅")
print(f"   Été   = solaire fort + éolien plus faible → complémentaires ✅")

✅ Index commun : 8784 heures



💡 Corrélation solaire/éolien : -0.222
   Valeur proche de 0 = pas de lien fort (normal solaire vs éolien)
   Hiver = éolien fort + solaire faible → complémentaires ✅
   Été   = solaire fort + éolien plus faible → complémentaires ✅


---
## 8. 📋 Résumé de qualité des données
Tableau de synthèse pour savoir quoi améliorer en priorité

In [14]:
summary_data = {
    'Fichier': [
        'solar_patterns.db',
        'wind_patterns.db',
        'gisdata/db_semippa_auto.csv',
        'grid_france.csv',
        'load_patterns.db',
        'TURPE_france.xlsx',
        'wind_grid_france.xlsx',
        'DVF (data.gouv.fr CSV)',
        'INPN zones protégées'
    ],
    'Statut': [
        '🟢 Réel',
        '🟢 Réel',
        '🟢 Réel',
        '🟡 Synthétique',
        '🔴 À collecter',
        '🟢 Réel',
        '🟡 Partiel',
        '🟢 Réel',
        '🟢 Réel'
    ],
    'Source': [
        'PVGIS JRC (lat/lon client, 2020)',
        'Open-Meteo ERA5 (5 régions offshore, 2020)',
        'IGN RPG.LATEST WFS (57 000 parcelles PAC, 2024)',
        'Projections ADEME/RTE + eco2mix',
        'Profil industriel réel à demander au client',
        'CRE TURPE 6 HTB Nov 2024 (délibération 2024-121)',
        '6 parcs encodés + AO CRE (à compléter)',
        'DGFiP via data.gouv.fr par département (2023)',
        'IGN Géoplateforme WFS (patrinat_znieff1/sic/zps)'
    ],
    'Priorité amélioration': [
        '🔵 Basse — données de qualité JRC',
        '🔵 Basse — ERA5 fiable, ajouter années si besoin',
        '🔵 Basse — 57 000 vraies parcelles PAC 2024',
        '🟠 Haute — télécharger eco2mix ODRE pour CO2 réel',
        '🔴 Critique — contacter le client dès que possible',
        '🔵 Basse — tarifs officiels CRE Nov 2024',
        '🟠 Haute — ajouter parcs AO CRE 2023 (Dunkerque 2...)',
        '🟡 Moyenne — prix à affiner (filtre terrains agricoles)',
        '🔵 Basse — OK, zones réelles INPN'
    ],
    'Où obtenir les données réelles': [
        'pvgis.ec.europa.eu — changer lat/lon dans download_france_data.py',
        'open-meteo.com — ajouter zones Méditerranée/Manche si besoin',
        'Automatique via build_land_grid.py (IGN WFS)',
        'odre.opendatasoft.com → eco2mix national → passer à build_france_grid_info()',
        'Client : export courbe de charge horodate (format CSV)',
        'cre.fr → délibérations → TURPE 6 HTB (déjà intégré dans TURPE_france.xlsx)',
        'cre.fr → AO éolien en mer + thewindpower.net + 4C Offshore',
        'Automatique via build_land_grid.py (fetch_dvf_from_csv)',
        'Automatique via build_land_grid.py (patrinat WFS IGN)'
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n📋 SYNTHÈSE QUALITÉ DES DONNÉES — French PPA Model\n")
pd.set_option('display.max_colwidth', 65)
display(summary_df.set_index('Fichier'))

# Compter les statuts
n_reel = summary_data['Statut'].count('🟢 Réel')
n_synth = summary_data['Statut'].count('🟡 Synthétique') + summary_data['Statut'].count('🟡 Partiel')
n_crit = summary_data['Statut'].count('🔴 À collecter')

print(f"\n📊 Résumé : {n_reel} fichiers réels ✅ | {n_synth} partiels ⚠️ | {n_crit} à collecter 🚨")
print("\n🎯 Actions prioritaires :")
print("   1. 🔴 [CRITIQUE] Courbe de charge réelle du client → contacter gestionnaire énergie")
print("   2. 🟠 [HAUTE]    eco2mix ODRE → odre.opendatasoft.com → grid_france.csv")
print("   3. 🟠 [HAUTE]    Parcs éoliens AO CRE → wind_grid_france.xlsx (Dunkerque 2, Manche 1...)")
print("   4. 🟡 [MOYENNE]  DVF prix agricoles → affiner filtre nature_culture dans build_land_grid.py")
print("   5. 🔵 [BASSE]    PVGIS autre site → changer lat/lon dans download_france_data.py")


📋 SYNTHÈSE QUALITÉ DES DONNÉES — French PPA Model



,Statut,Source,Priorité amélioration,Où obtenir les données réelles
Fichier,,,,
solar_patterns.db,🟢 Réel,"PVGIS JRC (lat/lon client, 2020)",🔵 Basse — données de qualité JRC,pvgis.ec.europa.eu — changer lat/lon dans download_france_dat...
wind_patterns.db,🟢 Réel,"Open-Meteo ERA5 (5 régions offshore, 2020)","🔵 Basse — ERA5 fiable, ajouter années si besoin",open-meteo.com — ajouter zones Méditerranée/Manche si besoin
gisdata/db_semippa_auto.csv,🟢 Réel,"IGN RPG.LATEST WFS (57 000 parcelles PAC, 2024)",🔵 Basse — 57 000 vraies parcelles PAC 2024,Automatique via build_land_grid.py (IGN WFS)
grid_france.csv,🟡 Synthétique,Projections ADEME/RTE + eco2mix,🟠 Haute — télécharger eco2mix ODRE pour CO2 réel,odre.opendatasoft.com → eco2mix national → passer à build_fra...
load_patterns.db,🔴 À collecter,Profil industriel réel à demander au client,🔴 Critique — contacter le client dès que possible,Client : export courbe de charge horodate (format CSV)
TURPE_france.xlsx,🟢 Réel,CRE TURPE 6 HTB Nov 2024 (délibération 2024-121),🔵 Basse — tarifs officiels CRE Nov 2024,cre.fr → délibérations → TURPE 6 HTB (déjà intégré dans TURPE...
wind_grid_france.xlsx,🟡 Partiel,6 parcs encodés + AO CRE (à compléter),🟠 Haute — ajouter parcs AO CRE 2023 (Dunkerque 2...),cre.fr → AO éolien en mer + thewindpower.net + 4C Offshore
DVF (data.gouv.fr CSV),🟢 Réel,DGFiP via data.gouv.fr par département (2023),🟡 Moyenne — prix à affiner (filtre terrains agricoles),Automatique via build_land_grid.py (fetch_dvf_from_csv)
INPN zones protégées,🟢 Réel,IGN Géoplateforme WFS (patrinat_znieff1/sic/zps),"🔵 Basse — OK, zones réelles INPN",Automatique via build_land_grid.py (patrinat WFS IGN)



📊 Résumé : 6 fichiers réels ✅ | 2 partiels ⚠️ | 1 à collecter 🚨

🎯 Actions prioritaires :
   1. 🔴 [CRITIQUE] Courbe de charge réelle du client → contacter gestionnaire énergie
   2. 🟠 [HAUTE]    eco2mix ODRE → odre.opendatasoft.com → grid_france.csv
   3. 🟠 [HAUTE]    Parcs éoliens AO CRE → wind_grid_france.xlsx (Dunkerque 2, Manche 1...)
   4. 🟡 [MOYENNE]  DVF prix agricoles → affiner filtre nature_culture dans build_land_grid.py
   5. 🔵 [BASSE]    PVGIS autre site → changer lat/lon dans download_france_data.py
